# 02 – Neuronales Netz mit PyTorch

**Lernziele:**
- `nn.Module` als Baustein für neuronale Netze
- `DataLoader` & `Dataset` für effizientes Daten-Handling
- `Optimizer` & `Loss`-Funktionen
- Train/Test-Loop mit GPU-Support
- MLP vs. CNN auf MNIST vergleichen

---

In [ ]:
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

print(f"PyTorch Version: {torch.__version__}")

## 1. Device wählen

Automatisch GPU nutzen, wenn verfügbar.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Daten laden: MNIST

MNIST: 70.000 handgeschriebene Ziffern (28×28 Graustufen), 10 Klassen (0–9).

`transforms.Normalize` zentriert die Daten mit dem bekannten MNIST-Mittelwert (0.1307) und Standardabweichung (0.3081).

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_data = datasets.MNIST("./data", train=True, download=True, transform=transform)
test_data = datasets.MNIST("./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1000)

print(f"Train: {len(train_data):,} | Test: {len(test_data):,}")

## 3. Modell 1: SimpleMLP

Ein **Multi-Layer Perceptron** (vollvernetztes Netz):
- Input: 784 (28×28 flachgeklopft)
- Hidden: 128 → 64 (ReLU + Dropout)
- Output: 10 (Logits für jede Ziffer)

**Wichtig:** Kein Softmax im Output – `CrossEntropyLoss` erwartet Logits und wendet Softmax intern an.

In [ ]:
class SimpleMLP(nn.Module):
    """784 → 128 → 64 → 10"""

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = x.view(x.size(0), -1)  # Flatten: (N, 28, 28) → (N, 784)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        return self.fc3(x)  # Logits

print(SimpleMLP())

## 4. Modell 2: SimpleCNN

Ein **Convolutional Neural Network**:
- Conv2d(1→32, 3×3) → ReLU → MaxPool(2×2) → 14×14
- Conv2d(32→64, 3×3) → ReLU → MaxPool(2×2) → 7×7
- Flatten → FC(64×7×7 → 128) → FC(128 → 10)

CNNs nutzen räumliche Struktur – ideal für Bilder!

In [ ]:
class SimpleCNN(nn.Module):
    """Conv → ReLU → MaxPool → Conv → ReLU → MaxPool → FC → FC"""

    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.25)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # → (N, 32, 14, 14)
        x = self.pool(F.relu(self.conv2(x)))  # → (N, 64, 7, 7)
        x = x.view(x.size(0), -1)             # Flatten
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

print(SimpleCNN())

## 5. Training & Evaluation

Standard-Trainingsloop:
1. `model.train()` – aktiviert Dropout/BatchNorm
2. `optimizer.zero_grad()` – Gradienten zurücksetzen
3. Forward-Pass
4. `loss.backward()` – Gradienten berechnen
5. `optimizer.step()` – Parameter updaten

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    """Eine Trainings-Epoche."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for data, target in loader:
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += target.size(0)

    return total_loss / len(loader), correct / total


def evaluate(model, loader, criterion, device):
    """Evaluation auf Testdaten."""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)

            total_loss += loss.item()
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()
            total += target.size(0)

    return total_loss / len(loader), correct / total


def count_parameters(model):
    """Zählt trainierbare Parameter."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 6. Modelle trainieren und vergleichen

Wir trainieren beide Architekturen für je 5 Epochen und vergleichen die Ergebnisse.

In [ ]:
models = {
    "MLP (784→128→64→10)": SimpleMLP(),
    "CNN (2xConv+2xFC)": SimpleCNN(),
}

for name, model in models.items():
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")

    model = model.to(device)
    print(f"   Parameter: {count_parameters(model):,}")

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    epochs = 5

    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(
            model, train_loader, optimizer, criterion, device
        )
        test_loss, test_acc = evaluate(
            model, test_loader, criterion, device
        )
        print(f"   Epoche {epoch+1}: "
              f"Train Loss={train_loss:.4f} Acc={train_acc:.3f} | "
              f"Test Loss={test_loss:.4f} Acc={test_acc:.3f}")

print("\n✅ Training abgeschlossen!")

---
## Zusammenfassung

| Konzept | Beschreibung |
|---|---|
| **`nn.Module`** | Basisklasse für alle neuronalen Netze |
| **`nn.Linear`** | Vollvernetzte Schicht (Matrix-Multiplikation + Bias) |
| **`nn.Conv2d`** | 2D-Faltung für räumliche Features |
| **`DataLoader`** | Batching, Shuffling, Multi-Processing |
| **`CrossEntropyLoss`** | Kombiniert LogSoftmax + NLLLoss |
| **`Adam`** | Adaptiver Optimizer (gute Default-Werte) |
| **`model.train()` / `model.eval()`** | Schaltet Dropout/BatchNorm um |

**Nächstes Notebook:** `03_cnn_mnist.ipynb` – Vertieftes CNN mit BatchNorm, Visualisierung und Metriken.